# U04_P — GenBank and Formats — Solutions
**Course:** Concepts of Bioinformatics (CBBI-501) · **Unit 4:** GenBank Sequence Database
**Type:** Practice Solutions notebook · **Time:** 50 min · **Difficulty:** ★★☆
**Goes with:** Lectures L7–L9, Practicals S4–S5
**Before this:** U04_C1, U04_C2

In [1]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

NOTEBOOK_ID = "U04_P"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

import Bio
print("Python", sys.version.split()[0], "| Biopython", Bio.__version__)

Python 3.11.18 | Biopython 1.88


### Ex 4.1 ★☆☆ — Extract CDS coordinates for human insulin
**Biology:** Human preproinsulin (INS) is encoded by NM_000207.3.  
**Task:** Load `data/genbank/ins_human.gb`. Find the CDS feature and store its 1-based start coordinate in `ins_cds_start` and 1-based end coordinate in `ins_cds_end`.  
**Expected:** `ins_cds_start` = 60, `ins_cds_end` = 392.

In [2]:
from Bio import SeqIO

ins_gb = DATA_DIR / "genbank" / "ins_human.gb"
ins_rec = SeqIO.read(ins_gb, "genbank")

ins_cds = next(f for f in ins_rec.features if f.type == "CDS")
ins_cds_start = int(ins_cds.location.start) + 1
ins_cds_end = int(ins_cds.location.end)

In [3]:
# Check — do not edit
assert ins_cds_start == 60, f"Expected 60, got {ins_cds_start}"
assert ins_cds_end == 392, f"Expected 392, got {ins_cds_end}"
print("Ex 4.1 passed")

Ex 4.1 passed


### Ex 4.2 ★☆☆ — Compute protein length from nucleotide coordinates
**Biology:** The mature protein length in amino acids equals $(N_{\text{nt}} - 3) / 3$.  
**Task:** Using `ins_cds_start` and `ins_cds_end`, calculate the length of the coding nucleotide sequence `nt_len` and the resulting protein length `aa_len`.  
**Expected:** `nt_len` = 333, `aa_len` = 110.

In [4]:
nt_len = ins_cds_end - ins_cds_start + 1
aa_len = (nt_len - 3) // 3

In [5]:
# Check — do not edit
assert nt_len == 333, f"Expected 333 nt, got {nt_len}"
assert aa_len == 110, f"Expected 110 aa, got {aa_len}"
print("Ex 4.2 passed")

Ex 4.2 passed


### Ex 4.3 ★☆☆ — Extract protein length for TP53 tumor suppressor
**Biology:** Tumor protein p53 (TP53) regulates the cell cycle and genomic stability.  
**Task:** Load `data/genbank/tp53_human.gb`. Find the CDS feature and calculate its translated protein length in `tp53_aa_len`.  
**Expected:** `tp53_aa_len` = 393.

In [6]:
tp53_gb = DATA_DIR / "genbank" / "tp53_human.gb"
tp53_rec = SeqIO.read(tp53_gb, "genbank")

tp53_cds = next(f for f in tp53_rec.features if f.type == "CDS")
tp53_aa_len = len(tp53_cds.qualifiers["translation"][0])

In [7]:
# Check — do not edit
assert tp53_aa_len == 393, f"Expected 393 aa for TP53, got {tp53_aa_len}"
print("Ex 4.3 passed")

Ex 4.3 passed


### Ex 4.4 ★★☆ — Translate CDS and confirm identity with official GenBank translation
**Biology:** Translating extracted coding nucleotides must exactly reproduce the curated protein annotation.  
**Task:** Extract the CDS nucleotide sequence of INS with `ins_cds.extract(ins_rec.seq)` into `ins_cds_dna`. Translate it with `to_stop=True` into `ins_trans`. Compare with `ins_cds.qualifiers['translation'][0]`.  
**Expected:** `is_identical` as boolean True.

In [8]:
ins_cds_dna = ins_cds.extract(ins_rec.seq)
ins_trans = str(ins_cds_dna.translate(to_stop=True))
is_identical = (ins_trans == ins_cds.qualifiers["translation"][0])

In [9]:
# Check — do not edit
assert is_identical is True, "Calculated translation does not match GenBank translation"
print("Ex 4.4 passed")

Ex 4.4 passed


### Ex 4.5 ★★☆ — Convert INS GenBank file to FASTA and verify file contents
**Biology:** Converting records to FASTA facilitates similarity searching with BLAST.  
**Task:** Write `ins_rec` to `OUT_DIR / 'ins_converted.fasta'` in `'fasta'` format using `SeqIO.write`. Read it back and store sequence length in `fasta_len`.  
**Expected:** `fasta_len` equals length of `ins_rec.seq`.

In [10]:
out_fasta_path = OUT_DIR / "ins_converted.fasta"
SeqIO.write(ins_rec, out_fasta_path, "fasta")
reloaded = SeqIO.read(out_fasta_path, "fasta")
fasta_len = len(reloaded.seq)

In [11]:
# Check — do not edit
assert fasta_len == len(ins_rec.seq), f"Expected {len(ins_rec.seq)}, got {fasta_len}"
print("Ex 4.5 passed")

Ex 4.5 passed


### Ex 4.6 ★★☆ — Check feature stripping during FASTA conversion
**Biology:** FASTA format does not support feature tables.  
**Task:** Store the number of features in `ins_rec` in `orig_feat_count`, and the number of features in `reloaded` in `fasta_feat_count`.  
**Expected:** `orig_feat_count` > 1, `fasta_feat_count` == 0.

In [12]:
orig_feat_count = len(ins_rec.features)
fasta_feat_count = len(reloaded.features)

In [13]:
# Check — do not edit
assert orig_feat_count > 1, "Original record should have features"
assert fasta_feat_count == 0, "FASTA record cannot contain features"
print("Ex 4.6 passed")

Ex 4.6 passed


### Ex 4.7 ★★★ — Inspect EMBL format two-letter line codes
**Biology:** EMBL/ENA records use line identifiers such as ID (Identification), AC (Accession), and FT (Feature Table).  
**Task:** Convert `ins_rec` into EMBL format string using `io.StringIO()`. Count how many lines begin with `'FT   '` and store in `ft_line_count`.  
**Expected:** `ft_line_count` as integer > 5.

In [14]:
import io

buf = io.StringIO()
SeqIO.write(ins_rec, buf, "embl")
embl_text = buf.getvalue()
ft_line_count = sum(1 for line in embl_text.splitlines() if line.startswith("FT   "))

In [15]:
# Check — do not edit
assert ft_line_count > 5, f"Expected > 5 FT lines, got {ft_line_count}"
print("Ex 4.7 passed")

Ex 4.7 passed


## 5. Reflection Questions
1. Why does converting a GenBank flat file to FASTA discard biological feature locations, and why is this conversion non-reversible?
2. What are the key governance differences between primary database submissions (GenBank/ENA/DDBJ) and secondary databases (RefSeq)?